In [ ]:
!pip install bertopic
!pip install gensim

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-ua1_aau1
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-ua1_aau1
  Resolved https://github.com/huggingface/transformers to commit 68abf9baba1a2dd3d24c67b861879f31b893f0bf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random

In [12]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [13]:
df = pd.read_csv("/content/vibe_coding_combined_translated.csv")

In [14]:
df = df[["full_text_translated", "image_url"]]

In [ ]:
def preprocess_tweet(text):
    #lower case text
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    #remove symbols
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize and remove stop words
    word_tokens = word_tokenize(text)
    # Remove stop words
    filtered_sentence = [w for w in word_tokens if not w in stop_words]
    # Join the filtered words back into a string
    filtered_text = ' '.join(filtered_sentence)

    return filtered_text


In [35]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [36]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

,0
full_text_translated,20863
image_url,6617


In [37]:
docs = df_preprocessed['full_text_translated'].tolist()
images  = df_preprocessed['image_url'].tolist()
for i in range(len(images)):
    if pd.isna(images[i]):
        images[i] = None
random.shuffle(docs)
docs[:10]

['i am debuting the new vibe coding studio right now would be honored if you came by and took a look it features my own virtual coding environment i built from scratch it includes code generation analysis and of course completion it gives users direct access to',
 'when you vibe code youre playing with virtual legos',
 'claude recommending to make private repos public because its easier people vibecoding who dont know better and always follow recommendations are ngmi',
 'is vibe coding really safe this cmu paper mainly studies agentgenerated code vulnerability benchmarking based on realworld tasks although ai agents are getting better and better in terms of functionality of code generation they are shockingly fragile in terms of security more than 80 of even functionally perfect code contains serious security vulnerabilities 背景什么是 vibe',
 'i vibe code with local models i am not like you especially i dont have any debt lol',
 'me vibecoding my way through migrating from assistant api to

In [38]:
# embedding_model = MultiModalBackend('clip-ViT-B-32', batch_size=32)
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# Embed both images and documents, then average them
# doc_image_embeddings = embedding_model.embed(docs, images)

umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0)
hdbscan_model = HDBSCAN(min_cluster_size=15, min_samples=1)
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,        # remove very frequent words
    min_df=2
)

# umap_model = UMAP(n_components=2, n_neighbors=20, metric='cosine', random_state=42)
# hdbscan_model = HDBSCAN(min_cluster_size=100, max_cluster_size=8000, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 1))

representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
topics, probs = topic_model.fit_transform(docs)
topics = topic_model.reduce_outliers(docs, topics)
topic_model.reduce_topics(docs, nr_topics=20)
topics = topic_model.topics_

In [748]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7646,-1_coding vibe_vibecode_vibecoding_vibe code,"[coding vibe, vibecode, vibecoding, vibe code,...","[vibecoding viability gt agent viability, buil..."
1,0,10714,0_vibecode_vibe code_coding vibe_code vibe,"[vibecode, vibe code, coding vibe, code vibe, ...",[if you are technical and super passionate abo...
2,1,918,1_dapps_saas_dapp_trading bot,"[dapps, saas, dapp, trading bot, crypto, monet...",[if you are using a saas that does not an ai y...
3,2,251,2_gemini pro_vibe code_coding gemini_code gemini,"[gemini pro, vibe code, coding gemini, code ge...",[killer vibe coding combo google gemini 25 pro...
4,3,116,3_vs code_github copilot_copilot agent_coding ...,"[vs code, github copilot, copilot agent, codin...",[見てる vibe coding with github copilot agent mod...
5,4,102,4_people interested_web dev_im looking_interes...,"[people interested, web dev, im looking, inter...",[im looking to with folks who are into javasc...
6,5,92,5_doctors_doctor_healthcare_medical,"[doctors, doctor, healthcare, medical, hospita...",[tech has seriously gotten the entire healthca...
7,6,80,6_musk_elon_elon musk_trump,"[musk, elon, elon musk, trump, propaganda, rhe...",[woke liberals are getting triggered by this e...
8,7,74,7_dashboards_dashboard_analytics_backend,"[dashboards, dashboard, analytics, backend, vi...",[about three months ago we launched dgi a dat...
9,8,72,8_coding kids_vibecode_vibe code_kids vibe,"[coding kids, vibecode, vibe code, kids vibe, ...",[fun start to christmas eve vibe coding this s...


In [749]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Get topic words
topic_words = topic_model.get_topics()
topic_words = [[word for word, _ in topic_model.get_topic(topic_id)] for topic_id in topic_model.get_topics().keys() if topic_id != -1]

# Prepare docs for coherence model
tokenized_docs = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

In [750]:
# Get topic words
topic_words_raw = [[word for word, _ in topic_model.get_topic(topic_id)] for topic_id in topic_model.get_topics().keys() if topic_id != -1]

# Process topic_words_raw to get individual words that exist in our dictionary
processed_topic_words = []
for topic_list in topic_words_raw:
    current_topic_words = []
    for word_or_ngram in topic_list:
        # Split n-grams into individual words
        individual_words = word_or_ngram.split()
        for word in individual_words:
            # Only add words that are in our dictionary
            if word in dictionary.token2id:
                current_topic_words.append(word)
    # Only append non-empty topic lists
    if current_topic_words:
        processed_topic_words.append(current_topic_words)

# Calculate C_V coherence
coherence_model_cv = CoherenceModel(topics=processed_topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
coherence_cv = coherence_model_cv.get_coherence()
print(f"Coherence (C_V): {coherence_cv}")

Coherence (C_V): 0.4620045354451822


In [751]:
# Calculate NPMI coherence
coherence_model_npmi = CoherenceModel(topics=processed_topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi')
coherence_npmi = coherence_model_npmi.get_coherence()
print(f"Coherence (NPMI): {coherence_npmi}")

Coherence (NPMI): 0.04977566787484498
